In [2]:
!pip install -U transformers peft accelerate bitsandbytes rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 10.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=fa02d1a9336967b3992ef578f2120bfc0e812437cbf6e610bf1abf94efb8034e
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate

In [37]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from rouge_score import rouge_scorer
import pandas as pd
from rouge_score import rouge_scorer

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_PATH = "/content/drive/MyDrive/Circle/lora-adapter"
EVAL_PATH = "/content/sample_data/eval.jsonl"

In [25]:
eval_instructions = [
    "Explain why the sky is blue.",
    "Give three tips for learning a new language.",
    "What is the capital of Japan?",
    "Write a function to check if a number is prime.",
    "Describe how a rainbow forms.",
    "Suggest three healthy breakfast ideas.",
    "What are the benefits of regular exercise?",
    "Explain the difference between a list and a tuple in Python.",
    "Give a short definition of machine learning.",
    "Suggest a name for a coffee shop.",
    "Write a function to reverse a string.",
    "What causes earthquakes?",
    "Give three tips for better sleep.",
    "Explain what an API is.",
    "Describe the water cycle."
]

with open(EVAL_PATH, "w", encoding="utf-8") as f:
    for instr in eval_instructions:
        f.write(json.dumps({"instruction": instr}, ensure_ascii=False) + "\n")

print(f"{len(eval_instructions)}")

15


In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map={"": 0}
)

tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [8]:
def generate(model, instruction):
    prompt = f"[INST] {instruction} [/INST]"
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    text = tok.decode(out[0], skip_special_tokens=True)
    return text.split("[/INST]")[-1].strip()

In [24]:
instruction = "What is the capital of Japan?"
base_answer = generate(base_model, instruction)
tuned_answer = generate(tuned_model, instruction)

print("model:")
print(base_answer)
print("\ntuned model:")
print(tuned_answer)

model:
Tokyo is Japan's capital city.

tuned model:
Tokyo is Japan's capital city.


In [15]:
base_predictions = []
tuned_predictions = []

for instruction in eval_instructions:

    base_predictions.append(
        generate(base_model, instruction)
    )
    tuned_predictions.append(
        generate(tuned_model, instruction)
    )

print("Base predictions:", len(base_predictions))
print("Tuned predictions:", len(tuned_predictions))

Base predictions: 15
Tuned predictions: 15


In [16]:
references = [
    "The sky appears blue because molecules in Earth's atmosphere scatter blue light more strongly than other visible wavelengths.",
    "Three tips for learning a new language are to practice regularly, learn vocabulary in context, and use the language through speaking, listening, reading, and writing.",
    "The capital of Japan is Tokyo.",
    "A number is prime if it is greater than 1 and has no positive divisors other than 1 and itself. In Python, you can test divisibility from 2 up to the square root of the number.",
    "A rainbow forms when sunlight enters water droplets and is refracted, reflected, and dispersed into different colors before reaching the observer.",
    "Three healthy breakfast ideas are oatmeal with fruit and nuts, eggs with whole-grain toast and vegetables, and yogurt with berries and oats.",
    "Regular exercise can improve cardiovascular fitness, strengthen muscles and bones, support mental well-being, and improve overall health.",
    "A Python list is mutable, meaning its elements can be changed, while a tuple is immutable. Lists are commonly written with square brackets and tuples with parentheses.",
    "Machine learning is a field of artificial intelligence where models learn patterns from data to make predictions or decisions without being explicitly programmed for every case.",
    "A possible name for a coffee shop is Morning Bean.",
    "A string can be reversed in Python using slicing, for example: reversed_string = text[::-1].",
    "Earthquakes are mainly caused by the sudden release of energy when tectonic plates move along faults in Earth's crust.",
    "Three tips for better sleep are keeping a regular sleep schedule, avoiding stimulating activities before bedtime, and keeping the bedroom quiet, dark, and comfortable.",
    "An API, or Application Programming Interface, is a set of rules that allows different software applications or services to communicate with each other.",
    "The water cycle is the continuous movement of water through processes such as evaporation, condensation, precipitation, and collection."
]

print("Instructions:", len(eval_instructions))
print("Base predictions:", len(base_predictions))
print("Tuned predictions:", len(tuned_predictions))
print("References:", len(references))

Instructions: 15
Base predictions: 15
Tuned predictions: 15
References: 15


In [17]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

In [20]:
def calculate_rouge(predictions, references):
    scores = {
        "rouge1": [],
        "rouge2": [],
        "rougeL": []
    }

    for prediction, reference in zip(predictions, references):
        score = scorer.score(
            reference,
            prediction
        )

        for metric in scores:
            scores[metric].append(
                score[metric].fmeasure
            )

    return {
        metric: sum(values) / len(values)
        for metric, values in scores.items()
    }

In [21]:
base_scores = calculate_rouge(
    base_predictions,
    references
)

tuned_scores = calculate_rouge(
    tuned_predictions,
    references
)

In [23]:
print("model")

for metric, score in base_scores.items():
    print(f"{metric}: {score:.4f}")

print("\ntuned model")

for metric, score in tuned_scores.items():
    print(f"{metric}: {score:.4f}")

model
rouge1: 0.3539
rouge2: 0.0806
rougeL: 0.2494

tuned model
rouge1: 0.3539
rouge2: 0.0806
rougeL: 0.2494


In [36]:
results = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Base Mistral-7B": [
        base_scores["rouge1"],
        base_scores["rouge2"],
        base_scores["rougeL"]
    ],

    "Mistral-7B + LoRA": [
        tuned_scores["rouge1"],
        tuned_scores["rouge2"],
        tuned_scores["rougeL"]
    ]
})

results

,Metric,Base Mistral-7B,Mistral-7B + LoRA
0,ROUGE-1,0.353861,0.353861
1,ROUGE-2,0.080595,0.080595
2,ROUGE-L,0.249414,0.249414


In [33]:
for i in range(15):
    print(f"---Example {i + 1}---")
    print("\nInstruction:")
    print(eval_instructions[i])

    print("\nreference:")
    print(references[i])

    print("\nbased:")
    print(base_predictions[i])

    print("\ntuned:")
    print(tuned_predictions[i])

---Example 1---

Instruction:
Explain why the sky is blue.

reference:
The sky appears blue because molecules in Earth's atmosphere scatter blue light more strongly than other visible wavelengths.

based:
Sky appears blue due to light scattering by Earth's atmosphere molecules. Sunlight interacts, causing light particles to scatter, with blue light more easily transmitted, resulting in its prevalence in our vision.

tuned:
Sky appears blue due to light scattering by Earth's atmosphere molecules. Sunlight interacts, causing light particles to scatter, with blue light more easily transmitted, resulting in its prevalence in our vision.
---Example 2---

Instruction:
Give three tips for learning a new language.

reference:
Three tips for learning a new language are to practice regularly, learn vocabulary in context, and use the language through speaking, listening, reading, and writing.

based:
1. Practice daily: study vocabulary, grammar rules, and listen/speak.
2. Immerse yourself: watch/

In [35]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def calculate_rouge(predictions, references):
    scores = {
        "rouge1": [],
        "rouge2": [],
        "rougeL": []
    }

    for prediction, reference in zip(predictions, references):

        score = scorer.score(
            reference,
            prediction
        )

        for metric in scores:
            scores[metric].append(
                score[metric].fmeasure
            )

    return {
        metric: sum(values) / len(values)
        for metric, values in scores.items()
    }


base_scores = calculate_rouge(
    base_predictions,
    references
)

tuned_scores = calculate_rouge(
    tuned_predictions,
    references
)

print("model")
for metric, score in base_scores.items():
    print(f"{metric}: {score:.4f}")

print("\ntuned model")
for metric, score in tuned_scores.items():
    print(f"{metric}: {score:.4f}")

model
rouge1: 0.3539
rouge2: 0.0806
rougeL: 0.2494

tuned model
rouge1: 0.3539
rouge2: 0.0806
rougeL: 0.2494
